# HCP-1200 Combined Benchmark Score Table

This notebook combines completed PySPI and skarf benchmark outputs into one estimator-by-metric table. Prediction benchmarks remain on the full-matrix `combined` rows, while directional `upper` and `lower` rows are added only for benchmarks that actually calculated directional scores.

## Set Up Imports and Configuration



Import the libraries used by the loaders and define benchmark toggles, configurable metric-rule mappings, result-directory defaults, output paths, and ranking options.

In [1]:
import json
import os
import re
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd

from arfcexp.matrices import load_symmetry_lookup
from arfcexp.skarf_utils import AVAILABLE_SKARF_FUNCS
from skarf import set_cache_dir
from skarf.covariance import load_spi_config_map

pd.options.display.precision = 4

PROJECT_ROOT = Path(
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
).resolve()

RESULT_ROOT = PROJECT_ROOT / "results"

ANALYSIS_DIR = Path(
    PROJECT_ROOT / "results/hcp_1200_benchmark_scores_combined_analysis",
).resolve()
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

KEY_COLS = ["method", "func", "lag", "component"]
COMPONENT_ORDER = ["combined", "upper", "lower"]
PYPSI_CATEGORY_ORDER = [
    "basic",
    "causal",
    "distance",
    "infotheory",
    "misc",
    "spectral",
    "wavelet",
    "unknown",
]
SPARSITY_TARGET = 0.8
PHENOTYPIC_TARGETS = ["Cognition"]
EXTRINSIC_BENCHMARKS = [
    "phenotypic_non_sparse",
    # "phenotypic_sparse",
    "demographics",
    # "homotopic",
    # "weight_distance",
]
INTRINSIC_BENCHMARKS = [
    "info_density",
    "reliability",
]
INFO_DENSITY_EXCLUDE = {
    # "sv_entropy",
    # "stable_rank",
    "tsp_cost",
    "small_worldness", 
    "trophic_incoherence",
    "trophic_incoherence_abs",
    "core_depth",
    "rich_club_auc",
    "rich_club_mean_rho",
    "rich_club_k_at_max",
    "rich_club_sig_k_range",
}
RELIABILITY_EXCLUDE = {
    # "mean_icc2",
    "mean_icc3",
    # "gradient_similarity",
    # "discriminability",
    "I_diff",
    "success_rate",
}
ENABLED_BENCHMARKS = EXTRINSIC_BENCHMARKS + INTRINSIC_BENCHMARKS
VALID_RULES = {"higher", "lower", "absolute"}
METRIC_RULE_CONFIG = {
    "phenotypic_non_sparse_corr": {
        "match": "regex",
        "patterns": [r"^pheno_(?!sparse_).+_corr$"],
        "rule": "higher",
        "weight": 0.5,
    },
    "phenotypic_non_sparse_r2": {
        "match": "regex",
        "patterns": [r"^pheno_(?!sparse_).+_r2$"],
        "rule": "higher",
        "weight": 0.5,
    },
    # "phenotypic_sparse_corr": {
    #     "match": "regex",
    #     "patterns": [r"^pheno_sparse_.+_corr$"],
    #     "rule": "higher",
    #     "weight": 0.5,
    # },
    # "phenotypic_sparse_r2": {
    #     "match": "regex",
    #     "patterns": [r"^pheno_sparse_.+_r2$"],
    #     "rule": "higher",
    #     "weight": 0.5,
    # },
    "demographics_sex": {
        "match": "exact",
        "patterns": ["demo_sex_acc"],
        "rule": "higher",
        "weight": 0.5,
    },
    "demographics_age_corr": {
        "match": "exact",
        "patterns": ["demo_age_corr"],
        "rule": "higher",
        "weight": 0.25,
    },
    "demographics_age_mae": {
        "match": "exact",
        "patterns": ["demo_age_mae"],
        "rule": "lower",
        "weight": 0.25,
    },
    # "homotopic": {
    #     "match": "exact",
    #     "patterns": ["homotopic_rank_mean"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
    # "weight_distance": {
    #     "match": "exact",
    #     "patterns": ["weight_distance_spearman"],
    #     "rule": "absolute",
    #     "weight": 1.0,
    # },
    # --- Intrinsic: reliability ---
    "reliability_icc2": {
        "match": "exact",
        "patterns": ["mean_icc2"],
        "rule": "higher",
        "weight": 0.33,
    },
    # "reliability_icc3": {
    #     "match": "exact",
    #     "patterns": ["mean_icc3"],
    #     "rule": "higher",
    #     "weight": 0.0,
    # },
    "reliability_gradient": {
        "match": "exact",
        "patterns": ["gradient_similarity"],
        "rule": "higher",
        "weight": 0.33,
    },
    "reliability_discriminability": {
        "match": "exact",
        "patterns": ["discriminability"],
        "rule": "higher",
        "weight": 0.33,
    },
    # "reliability_i_diff": {
    #     "match": "exact",
    #     "patterns": ["I_diff"],
    #     "rule": "lower",
    #     "weight": 1.0,
    # },
    #  "reliability_success_rate": {
    #     "match": "exact",
    #     "patterns": ["success_rate"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
    # --- Intrinsic: info_density ---
    "info_density_sv_entropy": {
        "match": "exact",
        "patterns": ["sv_entropy"],
        "rule": "higher",
        "weight": 0.5,
    },
    "info_density_stable_rank": {
        "match": "exact",
        "patterns": ["stable_rank"],
        "rule": "higher",
        "weight": 0.5,
    },
    # "info_density_tsp": {
    #     "match": "exact",
    #     "patterns": ["tsp_cost"],
    #     "rule": "lower",
    #     "weight": 1.0,
    # },
    # "info_density_small_worldness": {
    #     "match": "exact",
    #     "patterns": ["small_worldness"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
    # "info_density_trophic": {
    #     "match": "exact",
    #     "patterns": ["trophic_incoherence"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
    # "info_density_trophic_abs": {
    #     "match": "exact",
    #     "patterns": ["trophic_incoherence_abs"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
    # "info_density_rich_club_auc": {
    #     "match": "exact",
    #     "patterns": ["rich_club_auc"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
    # "info_density_rich_club_rho": {
    #     "match": "exact",
    #     "patterns": ["rich_club_mean_rho"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
    # "info_density_rich_club_k": {
    #     "match": "exact",
    #     "patterns": ["rich_club_k_at_max"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
    # "info_density_rich_club_sig": {
    #     "match": "exact",
    #     "patterns": ["rich_club_sig_k_range"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
    # "info_density_core_depth": {
    #     "match": "exact",
    #     "patterns": ["core_depth"],
    #     "rule": "higher",
    #     "weight": 1.0,
    # },
}
NON_METRIC_COLUMNS = {
    "method",
    "func",
    "lag",
    "component",
    "combo_key",
    "is_directed",
    "family",
    "display_name",
    "pyspi_category",
    "pyspi_identifier",
    "n_available_metrics",
    "missing_metrics",
    "rank_n_available",
    "minmax_rank_sum",
    "minmax_rank_order",
    "maxnorm_rank_sum",
    "maxnorm_rank_order",
    "quantile_rank_sum",
    "quantile_rank_order",
}

ALL_KNOWN_BENCHMARKS = EXTRINSIC_BENCHMARKS + INTRINSIC_BENCHMARKS
unknown_benchmarks = sorted(set(ENABLED_BENCHMARKS) - set(ALL_KNOWN_BENCHMARKS))
if unknown_benchmarks:
    raise ValueError(f"Unknown benchmark names in ENABLED_BENCHMARKS: {unknown_benchmarks}")
if not ENABLED_BENCHMARKS:
    raise ValueError("ENABLED_BENCHMARKS must include at least one benchmark.")

for rule_group, spec in METRIC_RULE_CONFIG.items():
    match_type = spec.get("match")
    patterns = spec.get("patterns", [])
    rule = spec.get("rule")
    weight = spec.get("weight")
    if match_type not in {"exact", "prefix", "regex"}:
        raise ValueError(f"Unsupported match type for {rule_group}: {match_type}")
    if not patterns:
        raise ValueError(f"METRIC_RULE_CONFIG[{rule_group!r}] must define at least one pattern.")
    if rule not in VALID_RULES:
        raise ValueError(f"Unsupported rule for {rule_group}: {rule}")
    if not isinstance(weight, (int, float)) or not np.isfinite(weight) or weight < 0:
        raise ValueError(f"METRIC_RULE_CONFIG[{rule_group!r}] must define a finite non-negative weight.")
    if match_type == "regex":
        for pattern in patterns:
            re.compile(str(pattern))

OUTPUT_RAW_PATH = ANALYSIS_DIR / "combined_benchmark_scores.csv"
OUTPUT_RANKED_PATH = ANALYSIS_DIR / "combined_benchmark_scores_ranked.csv"
OUTPUT_LONG_PATH = ANALYSIS_DIR / "combined_benchmark_scores_long.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ANALYSIS_DIR:", ANALYSIS_DIR)
print("Enabled benchmarks:", ", ".join(ENABLED_BENCHMARKS))
print("Metric rule groups:", ", ".join(METRIC_RULE_CONFIG))

PROJECT_ROOT: /home/aerkent/skarf-experiments
ANALYSIS_DIR: /home/aerkent/skarf-experiments/results/hcp_1200_benchmark_scores_combined_analysis
Enabled benchmarks: phenotypic_non_sparse, demographics, info_density, reliability
Metric rule groups: phenotypic_non_sparse_corr, phenotypic_non_sparse_r2, demographics_sex, demographics_age_corr, demographics_age_mae, reliability_icc2, reliability_gradient, reliability_discriminability, info_density_sv_entropy, info_density_stable_rank


## Load Project Inputs

Resolve benchmark result directories, load estimator metadata, and prepare the symmetry lookup used to decide which estimators are directed.

In [2]:
def resolve_result_dir(env_name: str, default_name: str) -> Path:
    if env_name in os.environ:
        return Path(os.environ[env_name]).expanduser().resolve()

    candidates = [RESULT_ROOT / default_name]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return candidates[0].resolve()


RESULT_DIRS = {
    "pheno_pyspi": resolve_result_dir(
        "PHENO_PYSPI_RESULT_DIR",
        "hcp_1200_pyspi_behav_prediction_factor_full",
    ),
    "pheno_skarf": resolve_result_dir(
        "PHENO_SKARF_RESULT_DIR",
        "hcp_1200_skarf_behav_prediction_factor_full",
    ),
    "pheno_sparse": resolve_result_dir(
        "SPARSE_PHENO_RESULT_DIR",
        "hcp_1200_sparse_behav_prediction",
    ),
    "demographics": resolve_result_dir(
        "DEMOGRAPHICS_RESULT_DIR",
        "hcp_1200_demographics_prediction",
    ),
    "pheno_ensemble": resolve_result_dir(
        "PHENO_ENSEMBLE_RESULT_DIR",
        "hcp_1200_ensemble_behav_prediction",
    ),
    "demographics_ensemble": resolve_result_dir(
        "DEMOGRAPHICS_ENSEMBLE_RESULT_DIR",
        "hcp_1200_demographics_prediction_ensemble",
    ),
    "homotopic": resolve_result_dir(
        "HOMOTOPIC_FC_RESULT_DIR",
        "hcp_1200_homotopic_fc_parallel",
    ),
    "weight_distance": resolve_result_dir(
        "WEIGHT_DISTANCE_RESULT_DIR",
        "hcp_1200_weight_distance_parallel",
    ),
}

INTRINSIC_PATHS = {
    "info_density": PROJECT_ROOT / "output" / "info_density_summary.csv",
    "reliability": PROJECT_ROOT / "output" / "reliability_summary.csv",
}

INTRINSIC_PATHS_ENSEMBLE = {
    "info_density": PROJECT_ROOT / "output" / "info_density_summary_ensemble.csv",
    "reliability": PROJECT_ROOT / "output" / "reliability_summary_ensemble.csv",
}

symmetry_lookup = load_symmetry_lookup(PROJECT_ROOT)

spi_list_dir = PROJECT_ROOT / "resources/spi_lists"
set_cache_dir(spi_list_dir)
try:
    spi_config_map, _ = load_spi_config_map()
except Exception as exc:
    spi_config_map = {}
    print(f"PySPI metadata load failed; continuing with basic labels: {exc}")

spi_order: list[str] = []
for list_name in ["spi_list_select_300s_149.txt", "spi_list_all_284.txt"]:
    list_path = spi_list_dir / list_name
    if list_path.exists():
        for spi in list_path.read_text().strip().split():
            if spi not in spi_order:
                spi_order.append(spi)

result_dir_table = pd.DataFrame(
    [
        {"benchmark": name, "path": str(path), "exists": path.exists()}
        for name, path in RESULT_DIRS.items()
    ]
    + [
        {"benchmark": name, "path": str(path), "exists": path.exists()}
        for name, path in INTRINSIC_PATHS.items()
    ]
)
display(result_dir_table)

,benchmark,path,exists
0,pheno_pyspi,/home/aerkent/skarf-experiments/results/hcp_12...,True
1,pheno_skarf,/home/aerkent/skarf-experiments/results/hcp_12...,True
2,pheno_sparse,/home/aerkent/skarf-experiments/results/hcp_12...,True
3,demographics,/home/aerkent/skarf-experiments/results/hcp_12...,True
4,pheno_ensemble,/home/aerkent/skarf-experiments/results/hcp_12...,True
5,demographics_ensemble,/home/aerkent/skarf-experiments/results/hcp_12...,True
6,homotopic,/home/aerkent/skarf-experiments/results/hcp_12...,True
7,weight_distance,/home/aerkent/skarf-experiments/results/hcp_12...,True
8,info_density,/home/aerkent/skarf-experiments/output/info_de...,True
9,reliability,/home/aerkent/skarf-experiments/output/reliabi...,True


In [3]:
# Load degenerate matrix lookup — applied in merge_metric_blocks to exclude
# any estimator whose FC matrices are degenerate (key format: "{method}__{func}")
degenerate_lookup_path = PROJECT_ROOT / "resources/matrix_degenerate_lookup.json"
degenerate_lookup = pd.read_json(degenerate_lookup_path, typ="series")
degenerate_keys = set(degenerate_lookup[degenerate_lookup].index)
print(f"Degenerate combos that will be excluded: {degenerate_keys}")


Degenerate combos that will be excluded: {'pyspi__lmfit_Lasso', 'pyspi__bary_euclidean_mean'}


## Define Core Data Structures

Define row keys, estimator metadata helpers, and metric-orientation rules that are reused by every benchmark loader.

In [4]:
def sanitize_label(value: object) -> str:
    text = str(value)
    text = re.sub(r"[^0-9A-Za-z]+", "_", text).strip("_")
    return text or "unknown"


def normalize_lag(value: object) -> int:
    if pd.isna(value):
        return 0
    return int(value)


def combo_key(method: str, func: str, lag: int) -> str:
    if method == "skarf":
        return f"{method}__{func}__lag-{int(lag)}"
    return f"{method}__{func}"


def is_directed_estimator(method: str, func: str) -> bool:
    key = f"{method}__{func}"
    if key not in symmetry_lookup:
        return False
    return not bool(symmetry_lookup[key])


def pyspi_category(func: str) -> str:
    config = spi_config_map.get(func, {})
    module_name = config.get("module_name", "unknown")
    category = module_name.split(".")[-1] if module_name != "unknown" else "unknown"
    if "wavelet" in func.lower():
        category = "wavelet"
    return category


def pyspi_identifier(func: str) -> str:
    return func.split("_", 1)[0] if "_" in func else func


def display_estimator(method: str, func: str, lag: int, component: str) -> str:
    if method == "skarf":
        base = func.replace("cov_", "cov:").replace("prec_", "prec:").replace("linear_", "lin:")
        label = f"skarf {base} lag {int(lag)}"
    elif method == "ensemble":
        label = f"Ensemble {func}"
    else:
        label = f"PySPI {func}"
    if component != "combined":
        label = f"{label} ({component})"
    return label


def add_row_metadata(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    out = df.copy()
    out["lag"] = out["lag"].fillna(0).astype(int)
    out["component"] = out["component"].fillna("combined").astype(str)
    out["family"] = np.select(
        [out["method"].eq("skarf"), out["method"].eq("ensemble")],
        ["skarf", "ensemble"],
        default="pyspi",
    )
    out["combo_key"] = [
        combo_key(row.method, row.func, row.lag) for row in out.itertuples()
    ]
    out["is_directed"] = [
        is_directed_estimator(row.method, row.func) for row in out.itertuples()
    ]
    out["display_name"] = [
        display_estimator(row.method, row.func, row.lag, row.component)
        for row in out.itertuples()
    ]
    out["pyspi_category"] = np.where(
        out["method"].eq("pyspi"),
        out["func"].map(pyspi_category),
        np.nan,
    )
    out["pyspi_identifier"] = np.where(
        out["method"].eq("pyspi"),
        out["func"].map(pyspi_identifier),
        np.nan,
    )
    return out


def estimator_sort_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    spi_rank = {func: idx for idx, func in enumerate(spi_order)}
    skarf_rank = {func: idx for idx, func in enumerate(AVAILABLE_SKARF_FUNCS)}
    component_rank = {component: idx for idx, component in enumerate(COMPONENT_ORDER)}
    category_rank = {category: idx for idx, category in enumerate(PYPSI_CATEGORY_ORDER)}

    out["_method_rank"] = out["method"].map({"pyspi": 0, "skarf": 1}).fillna(99).astype(int)
    out["_category_rank"] = out["pyspi_category"].map(category_rank).fillna(99).astype(int)
    out["_func_rank"] = np.where(
        out["method"].eq("skarf"),
        out["func"].map(skarf_rank).fillna(9999),
        out["func"].map(spi_rank).fillna(9999),
    ).astype(int)
    out["_component_rank"] = out["component"].map(component_rank).fillna(99).astype(int)
    out["_lag_rank"] = out["lag"].fillna(0).astype(int)
    return out


def drop_sort_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[col for col in df.columns if col.startswith("_")], errors="ignore")

## Implement Main Processing Function

The functions below load each benchmark family, convert scores to a shared row schema, and merge metric blocks into one wide table.

In [5]:
def parse_result_path(path: Path) -> dict[str, object]:
    parts = path.parts
    parsed: dict[str, object] = {"method": None, "func": None, "lag": None, "target": None}

    for part in parts:
        if part.startswith("target-"):
            parsed["target"] = part.removeprefix("target-")

        pyspi_match = re.match(r"^\d+__spi-(.+)$", part)
        if pyspi_match:
            parsed["method"] = "pyspi"
            parsed["func"] = pyspi_match.group(1)
            parsed["lag"] = 0

        skarf_match = re.match(r"^\d+__func-(.+?)(?:__lag-(\d+))?$", part)
        if skarf_match:
            parsed["method"] = "skarf"
            parsed["func"] = skarf_match.group(1)
            if skarf_match.group(2) is not None:
                parsed["lag"] = int(skarf_match.group(2))

        unified_match = re.match(r"^\d+__(pyspi|skarf|ensemble)__(.+?)(?:__lag-(\d+))?$", part)
        if unified_match:
            parsed["method"] = unified_match.group(1)
            parsed["func"] = unified_match.group(2)
            parsed["lag"] = int(unified_match.group(3) or 0)

        lag_match = re.search(r"(?:^|__)lag-(\d+)(?:__|$)", part)
        if lag_match and parsed["method"] == "skarf" and parsed["lag"] is None:
            parsed["lag"] = int(lag_match.group(1))

    if parsed["method"] == "pyspi":
        parsed["lag"] = 0
    if parsed["lag"] is None:
        parsed["lag"] = 0
    return parsed


def read_json_result_tree(root: Path) -> tuple[pd.DataFrame, list[dict]]:
    frames = []
    failures = []
    if not root.exists():
        return pd.DataFrame(), [{"path": str(root), "error": "missing result directory"}]

    for path in sorted(root.glob("**/results.json")):
        try:
            frame = pd.read_json(path, lines=True)
        except ValueError as exc:
            failures.append({"path": str(path), "error": str(exc)})
            continue
        if frame.empty:
            continue

        parsed = parse_result_path(path)
        for col, value in parsed.items():
            if col not in frame.columns or frame[col].isna().all():
                frame[col] = value
        frame["source_path"] = str(path)
        frames.append(frame)

    if not frames:
        return pd.DataFrame(), failures

    out = pd.concat(frames, ignore_index=True)
    out["lag"] = out["lag"].fillna(0).astype(int)
    return out, failures


def filter_non_permutation(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "perm_test" not in df.columns:
        return df.copy()
    return df.loc[~df["perm_test"].fillna(False).astype(bool)].copy()


def filter_prediction_targets(df: pd.DataFrame, targets: list[str]) -> pd.DataFrame:
    if df.empty or "target" not in df.columns:
        return df.copy()
    return df.loc[df["target"].astype(str).isin(targets)].copy()


def prediction_wide_from_results(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=KEY_COLS)

    required = {"method", "func", "lag", "target", "corr_test", "r2_test"}
    missing = sorted(required.difference(df.columns))
    if missing:
        raise KeyError(f"Prediction results missing required columns: {missing}")

    work = filter_non_permutation(df).dropna(subset=["method", "func", "target"]).copy()
    work["lag"] = work["lag"].fillna(0).astype(int)
    agg = (
        work.groupby(["method", "func", "lag", "target"], observed=True)
        .agg(
            corr=("corr_test", "mean"),
            r2=("r2_test", "mean"),
            n_splits=("split", "count"),
        )
        .reset_index()
    )

    value_frames = []
    for metric in ["corr", "r2"]:
        metric_frame = agg[["method", "func", "lag", "target", metric]].copy()
        metric_frame["metric_col"] = [
            f"{prefix}_{sanitize_label(target)}_{metric}"
            for target in metric_frame["target"]
        ]
        metric_frame.rename(columns={metric: "value"}, inplace=True)
        value_frames.append(metric_frame[["method", "func", "lag", "metric_col", "value"]])

    long = pd.concat(value_frames, ignore_index=True)
    wide = (
        long.pivot_table(
            index=["method", "func", "lag"],
            columns="metric_col",
            values="value",
            aggfunc="mean",
        )
        .reset_index()
        .rename_axis(columns=None)
    )
    wide["component"] = "combined"
    return wide


def load_phenotypic_prediction_blocks() -> tuple[list[pd.DataFrame], pd.DataFrame]:
    pyspi_results, pyspi_failures = read_json_result_tree(RESULT_DIRS["pheno_pyspi"])
    skarf_results, skarf_failures = read_json_result_tree(RESULT_DIRS["pheno_skarf"])
    sparse_results, sparse_failures = read_json_result_tree(RESULT_DIRS["pheno_sparse"])
    ensemble_results, ensemble_failures = read_json_result_tree(RESULT_DIRS["pheno_ensemble"])

    nonsparse_frames = [frame for frame in [pyspi_results, skarf_results, ensemble_results] if not frame.empty]
    nonsparse_results = pd.concat(nonsparse_frames, ignore_index=True) if nonsparse_frames else pd.DataFrame()
    nonsparse_results = filter_prediction_targets(nonsparse_results, PHENOTYPIC_TARGETS)

    nonsparse_block = prediction_wide_from_results(nonsparse_results, "pheno")

    sparse_block = pd.DataFrame(columns=KEY_COLS)
    if not sparse_results.empty:
        sparse_work = filter_non_permutation(sparse_results)
        sparse_work = filter_prediction_targets(sparse_work, PHENOTYPIC_TARGETS)
        if "sparsity" in sparse_work.columns:
            sparse_work = sparse_work.loc[np.isclose(sparse_work["sparsity"].astype(float), SPARSITY_TARGET)].copy()
        sparse_block = prediction_wide_from_results(sparse_work, "pheno_sparse")

    failures = pd.DataFrame(pyspi_failures + skarf_failures + sparse_failures + ensemble_failures)
    return [nonsparse_block, sparse_block], failures

In [6]:
def load_demographic_prediction_block() -> tuple[pd.DataFrame, pd.DataFrame]:
    core_results, core_failures = read_json_result_tree(RESULT_DIRS["demographics"])
    ensemble_results, ensemble_failures = read_json_result_tree(RESULT_DIRS["demographics_ensemble"])
    frames = [frame for frame in [core_results, ensemble_results] if not frame.empty]
    results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    failures = core_failures + ensemble_failures
    if results.empty:
        return pd.DataFrame(columns=KEY_COLS), pd.DataFrame(failures)

    work = filter_non_permutation(results).dropna(subset=["method", "func", "task"]).copy()
    work["lag"] = work["lag"].fillna(0).astype(int)
    blocks = []

    gender = work.loc[work["task"].eq("gender")].copy()
    if not gender.empty and "acc_test" in gender.columns:
        gender_block = (
            gender.groupby(["method", "func", "lag"], observed=True)
            .agg(demo_sex_acc=("acc_test", "mean"))
            .reset_index()
        )
        blocks.append(gender_block)

    age = work.loc[work["task"].eq("age")].copy()
    if not age.empty:
        age_aggs = {}
        if "corr_test" in age.columns:
            age_aggs["demo_age_corr"] = ("corr_test", "mean")
        if "mae_test" in age.columns:
            age_aggs["demo_age_mae"] = ("mae_test", "mean")
        if age_aggs:
            age_block = age.groupby(["method", "func", "lag"], observed=True).agg(**age_aggs).reset_index()
            blocks.append(age_block)

    if not blocks:
        return pd.DataFrame(columns=KEY_COLS), pd.DataFrame(failures)

    out = reduce(
        lambda left, right: left.merge(right, on=["method", "func", "lag"], how="outer"),
        blocks,
    )
    out["component"] = "combined"
    return out, pd.DataFrame(failures)


def summary_csv_paths(root: Path, filename: str) -> list[Path]:
    if not root.exists():
        return []
    if (root / filename).exists():
        return [root / filename]
    return sorted(path for path in root.glob(f"parc-*/{filename}") if path.exists())


def load_summary_csvs(root: Path, filename: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames = []
    failures = []
    for path in summary_csv_paths(root, filename):
        try:
            frame = pd.read_csv(path)
        except Exception as exc:
            failures.append({"path": str(path), "error": str(exc)})
            continue
        if frame.empty:
            continue
        frame["source_path"] = str(path)
        frames.append(frame)

    if not frames:
        if not root.exists():
            failures.append({"path": str(root), "error": "missing result directory"})
        else:
            failures.append({"path": str(root), "error": f"no {filename} files found"})
        return pd.DataFrame(), pd.DataFrame(failures)

    out = pd.concat(frames, ignore_index=True)
    if {"method", "func", "lag"}.issubset(out.columns):
        out["lag"] = out["lag"].fillna(0).astype(int)
        out = out.sort_values(["method", "func", "lag", "source_path"]).drop_duplicates(
            subset=["method", "func", "lag"],
            keep="last",
        )
    return out.reset_index(drop=True), pd.DataFrame(failures)


def load_homotopic_block() -> tuple[pd.DataFrame, pd.DataFrame]:
    summary, failures = load_summary_csvs(
        RESULT_DIRS["homotopic"],
        "method_homotopic_summary.csv",
    )
    if summary.empty:
        return pd.DataFrame(columns=KEY_COLS), failures

    rows = []
    for row in summary.itertuples(index=False):
        method = getattr(row, "method")
        func = getattr(row, "func")
        lag = normalize_lag(getattr(row, "lag", 0))
        is_directed = bool(getattr(row, "is_directed", is_directed_estimator(method, func)))

        mean = getattr(row, "mean", np.nan)
        if np.isfinite(mean):
            rows.append({
                "method": method,
                "func": func,
                "lag": lag,
                "component": "combined",
                "homotopic_rank_mean": float(mean),
            })

        if is_directed:
            for component, col in [("upper", "mean_upper"), ("lower", "mean_lower")]:
                value = getattr(row, col, np.nan)
                if np.isfinite(value):
                    rows.append({
                        "method": method,
                        "func": func,
                        "lag": lag,
                        "component": component,
                        "homotopic_rank_mean": float(value),
                    })

    return pd.DataFrame(rows), failures


def load_weight_distance_block() -> tuple[pd.DataFrame, pd.DataFrame]:
    summary, failures = load_summary_csvs(
        RESULT_DIRS["weight_distance"],
        "method_weight_distance_summary.csv",
    )
    if summary.empty:
        return pd.DataFrame(columns=KEY_COLS), failures

    rows = []
    for row in summary.itertuples(index=False):
        method = getattr(row, "method")
        func = getattr(row, "func")
        lag = normalize_lag(getattr(row, "lag", 0))
        is_directed = bool(getattr(row, "is_directed", is_directed_estimator(method, func)))

        combined = getattr(row, "score_mean", np.nan)
        if is_directed:
            combined = getattr(row, "score_offdiag_mean", combined)
        if np.isfinite(combined):
            rows.append({
                "method": method,
                "func": func,
                "lag": lag,
                "component": "combined",
                "weight_distance_spearman": float(combined),
            })

        if is_directed:
            for component, col in [("upper", "score_upper_mean"), ("lower", "score_lower_mean")]:
                value = getattr(row, col, np.nan)
                if np.isfinite(value):
                    rows.append({
                        "method": method,
                        "func": func,
                        "lag": lag,
                        "component": component,
                        "weight_distance_spearman": float(value),
                    })

    return pd.DataFrame(rows), failures


def load_intrinsic_csv_block(
    path: Path, exclude_cols: set[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load an intrinsic benchmark summary CSV into a metric block.

    Reads the CSV, drops excluded and non-metric metadata columns,
    normalizes the lag column, and sets component='combined' for all rows.
    """
    failures: list[dict] = []
    if not path.exists():
        failures.append({"path": str(path), "error": "file not found"})
        return pd.DataFrame(columns=KEY_COLS), pd.DataFrame(failures)

    try:
        raw = pd.read_csv(path)
    except Exception as exc:
        failures.append({"path": str(path), "error": str(exc)})
        return pd.DataFrame(columns=KEY_COLS), pd.DataFrame(failures)

    if raw.empty:
        failures.append({"path": str(path), "error": "empty CSV"})
        return pd.DataFrame(columns=KEY_COLS), pd.DataFrame(failures)

    # Normalize lag: empty string or NaN → 0, float → int
    raw["lag"] = pd.to_numeric(raw["lag"], errors="coerce").fillna(0).astype(int)
    raw["component"] = "combined"

    # Drop non-key metadata columns that came from the CSV
    csv_meta_cols = {"combo_key", "is_directed", "source_path"}
    drop_cols = exclude_cols | csv_meta_cols
    raw = raw.drop(columns=[col for col in drop_cols if col in raw.columns], errors="ignore")

    # Keep only KEY_COLS + numeric metric columns
    keep_cols = KEY_COLS + [
        col for col in raw.columns
        if col not in KEY_COLS and pd.api.types.is_numeric_dtype(raw[col])
    ]
    out = raw[keep_cols].copy()
    return out, pd.DataFrame(failures)


def load_info_density_block() -> tuple[pd.DataFrame, pd.DataFrame]:
    core_block, core_failures = load_intrinsic_csv_block(INTRINSIC_PATHS["info_density"], INFO_DENSITY_EXCLUDE)
    ensemble_block, ensemble_failures = load_intrinsic_csv_block(
        INTRINSIC_PATHS_ENSEMBLE["info_density"], INFO_DENSITY_EXCLUDE
    )
    frames = [frame for frame in [core_block, ensemble_block] if not frame.empty]
    block = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=KEY_COLS)
    failures = pd.concat([core_failures, ensemble_failures], ignore_index=True)
    return block, failures


def load_reliability_block() -> tuple[pd.DataFrame, pd.DataFrame]:
    core_block, core_failures = load_intrinsic_csv_block(INTRINSIC_PATHS["reliability"], RELIABILITY_EXCLUDE)
    ensemble_block, ensemble_failures = load_intrinsic_csv_block(
        INTRINSIC_PATHS_ENSEMBLE["reliability"], RELIABILITY_EXCLUDE
    )
    frames = [frame for frame in [core_block, ensemble_block] if not frame.empty]
    block = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=KEY_COLS)
    failures = pd.concat([core_failures, ensemble_failures], ignore_index=True)
    return block, failures

In [7]:
def collapse_metric_block(block: pd.DataFrame) -> pd.DataFrame:
    if block.empty:
        return block.copy()
    metric_cols = [col for col in block.columns if col not in KEY_COLS]
    block = block.copy()
    block["lag"] = block["lag"].fillna(0).astype(int)
    block["component"] = block["component"].fillna("combined")
    return (
        block.groupby(KEY_COLS, as_index=False, observed=True)[metric_cols]
        .mean(numeric_only=True)
        .reset_index(drop=True)
    )


def merge_metric_blocks(blocks: list[pd.DataFrame]) -> pd.DataFrame:
    valid_blocks = [collapse_metric_block(block) for block in blocks if block is not None and not block.empty]
    if not valid_blocks:
        return pd.DataFrame(columns=KEY_COLS)

    merged = reduce(
        lambda left, right: left.merge(right, on=KEY_COLS, how="outer"),
        valid_blocks,
    )
    merged["lag"] = merged["lag"].fillna(0).astype(int)

    # Exclude degenerate matrices
    lookup_keys = merged["method"] + "__" + merged["func"]
    mask_degenerate = lookup_keys.isin(degenerate_keys)
    if mask_degenerate.any():
        excluded = merged.loc[mask_degenerate, ["method", "func"]].drop_duplicates().values.tolist()
        print(f"Excluded degenerate combos from benchmark table: {excluded}")
        merged = merged[~mask_degenerate].copy()

    merged = add_row_metadata(merged)

    ordered_meta = [
        "method",
        "func",
        "lag",
        "component",
        "combo_key",
        "is_directed",
        "family",
        "display_name",
        "pyspi_category",
        "pyspi_identifier",
    ]
    metric_cols = [col for col in merged.columns if col not in ordered_meta]
    merged = merged[ordered_meta + sorted(metric_cols)]
    merged = estimator_sort_frame(merged).sort_values(
        ["_method_rank", "_category_rank", "_func_rank", "_lag_rank", "_component_rank", "func"]
    )
    return drop_sort_columns(merged).reset_index(drop=True)


def is_metric_column(metric_col: str) -> bool:
    return metric_col not in NON_METRIC_COLUMNS


def resolve_metric_rule_config(metric_col: str) -> dict[str, object] | None:
    if not is_metric_column(metric_col):
        return None

    match_priority = {"exact": 2, "regex": 1, "prefix": 0}
    matches = []
    for rule_group, spec in METRIC_RULE_CONFIG.items():
        match_type = str(spec["match"])
        patterns = [str(pattern) for pattern in spec["patterns"]]
        for pattern in patterns:
            matched = match_type == "exact" and metric_col == pattern
            matched = matched or (match_type == "prefix" and metric_col.startswith(pattern))
            matched = matched or (match_type == "regex" and re.fullmatch(pattern, metric_col) is not None)
            if matched:
                matches.append((match_priority[match_type], len(pattern), rule_group, spec, pattern))

    if not matches:
        return None

    matches.sort(key=lambda item: (item[0], item[1]), reverse=True)
    best = matches[0]
    if len(matches) > 1:
        second = matches[1]
        if (best[0], best[1]) == (second[0], second[1]) and best[2] != second[2]:
            raise ValueError(
                f"Ambiguous metric rule match for {metric_col}: "
                f"{best[2]} ({best[4]}) vs {second[2]} ({second[4]})"
            )

    return {
        "rule_group": best[2],
        "rule": str(best[3]["rule"]),
        "weight": float(best[3]["weight"]),
        "match": str(best[3]["match"]),
        "matched_pattern": best[4],
    }


def infer_metric_rule(metric_col: str) -> str | None:
    rule_info = resolve_metric_rule_config(metric_col)
    if rule_info is None:
        return None
    return str(rule_info["rule"])


def orient_values(values: pd.Series, rule: str) -> pd.Series:
    values = values.astype(float)
    if rule == "higher":
        return values
    if rule == "lower":
        return -values
    if rule in {"abs", "absolute"}:
        return values.abs()
    raise ValueError(f"Unsupported metric rule: {rule}")


def minmax_scale(values: pd.Series) -> pd.Series:
    out = pd.Series(np.nan, index=values.index, dtype=float)
    finite = values.replace([np.inf, -np.inf], np.nan).dropna()
    if finite.empty:
        return out
    min_value = finite.min()
    max_value = finite.max()
    if np.isclose(max_value, min_value):
        out.loc[finite.index] = 0.5
    else:
        out.loc[finite.index] = (finite - min_value) / (max_value - min_value)
    return out


def maxnorm_scale(values: pd.Series) -> pd.Series:
    out = pd.Series(np.nan, index=values.index, dtype=float)
    finite = values.replace([np.inf, -np.inf], np.nan).dropna()
    if finite.empty:
        return out
    if finite.nunique() == 1:
        out.loc[finite.index] = 1.0
        return out

    best_value = finite.max()
    if best_value <= 0 or np.isclose(best_value, 0.0):
        return minmax_scale(values)

    out.loc[finite.index] = finite / best_value
    return out


def quantile_score(values: pd.Series) -> pd.Series:
    out = pd.Series(np.nan, index=values.index, dtype=float)
    finite = values.replace([np.inf, -np.inf], np.nan).dropna()
    n_finite = len(finite)
    if n_finite == 0:
        return out

    sorted_index = finite.rank(method="first").sort_values().index
    scores = np.floor(np.arange(n_finite) * 4 / n_finite).clip(0, 3).astype(float)
    out.loc[sorted_index] = scores
    return out


def add_ranking_columns(table: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    ranked = table.copy()
    metric_cols = [col for col in table.columns if is_metric_column(col)]
    metric_rule_info = []
    for col in metric_cols:
        rule_info = resolve_metric_rule_config(col)
        if rule_info is None:
            raise ValueError(f"No ranking rule configured for metric column: {col}")
        metric_rule_info.append({"metric": col, **rule_info})

    for info in metric_rule_info:
        col = str(info["metric"])
        rule = str(info["rule"])
        oriented = orient_values(ranked[col], rule)
        ranked[f"minmax_{col}"] = minmax_scale(oriented)
        ranked[f"maxnorm_{col}"] = maxnorm_scale(oriented)
        ranked[f"qrank_{col}"] = quantile_score(oriented)

    metric_weights = {str(info["metric"]): float(info["weight"]) for info in metric_rule_info}
    ranked["rank_n_available"] = ranked[metric_cols].notna().sum(axis=1)
    ranked["minmax_rank_sum"] = sum(
        ranked[f"minmax_{col}"].fillna(0.0) * metric_weights[col] for col in metric_cols
    )
    ranked["maxnorm_rank_sum"] = sum(
        ranked[f"maxnorm_{col}"].fillna(0.0) * metric_weights[col] for col in metric_cols
    )
    ranked["quantile_rank_sum"] = sum(
        ranked[f"qrank_{col}"].fillna(0.0) * metric_weights[col] for col in metric_cols
    )
    ranked["minmax_rank_order"] = ranked["minmax_rank_sum"].rank(method="min", ascending=False).astype(int)
    ranked["maxnorm_rank_order"] = ranked["maxnorm_rank_sum"].rank(method="min", ascending=False).astype(int)
    ranked["quantile_rank_order"] = ranked["quantile_rank_sum"].rank(method="min", ascending=False).astype(int)

    metric_spec = pd.DataFrame(
        [
            {
                **info,
                "minmax_col": f"minmax_{info['metric']}",
                "maxnorm_col": f"maxnorm_{info['metric']}",
                "qrank_col": f"qrank_{info['metric']}",
            }
            for info in metric_rule_info
        ]
    ).sort_values("metric")
    return ranked, metric_spec.reset_index(drop=True)


## Run a Minimal Working Example



Load the enabled benchmark summaries, merge them into a wide score table, and add all configured ranking variants.

In [8]:
enabled_benchmarks = set(ENABLED_BENCHMARKS)
empty_block = pd.DataFrame(columns=KEY_COLS)
empty_failures = pd.DataFrame(columns=["path", "error"])

phenotypic_blocks = [empty_block.copy(), empty_block.copy()]
phenotypic_failures = empty_failures.copy()
if {"phenotypic_non_sparse", "phenotypic_sparse"} & enabled_benchmarks:
    phenotypic_blocks, phenotypic_failures = load_phenotypic_prediction_blocks()

demographics_block = empty_block.copy()
demographics_failures = empty_failures.copy()
if "demographics" in enabled_benchmarks:
    demographics_block, demographics_failures = load_demographic_prediction_block()

homotopic_block = empty_block.copy()
homotopic_failures = empty_failures.copy()
if "homotopic" in enabled_benchmarks:
    homotopic_block, homotopic_failures = load_homotopic_block()

weight_distance_block = empty_block.copy()
weight_distance_failures = empty_failures.copy()
if "weight_distance" in enabled_benchmarks:
    weight_distance_block, weight_distance_failures = load_weight_distance_block()

info_density_block = empty_block.copy()
info_density_failures = empty_failures.copy()
if "info_density" in enabled_benchmarks:
    info_density_block, info_density_failures = load_info_density_block()

reliability_block = empty_block.copy()
reliability_failures = empty_failures.copy()
if "reliability" in enabled_benchmarks:
    reliability_block, reliability_failures = load_reliability_block()

benchmark_blocks = {
    "phenotypic_non_sparse": phenotypic_blocks[0],
    "phenotypic_sparse": phenotypic_blocks[1],
    "demographics": demographics_block,/
}
metric_blocks = [benchmark_blocks[name] for name in ALL_KNOWN_BENCHMARKS if name in enabled_benchmarks]

combined_scores = merge_metric_blocks(metric_blocks)
metric_cols = [col for col in combined_scores.columns if is_metric_column(col)]
combined_scores["n_available_metrics"] = combined_scores[metric_cols].notna().sum(axis=1)
combined_scores["missing_metrics"] = [
    ", ".join(col for col in metric_cols if pd.isna(row[col]))
    for _, row in combined_scores.iterrows()
]

ranked_scores, metric_spec = add_ranking_columns(combined_scores)

failure_frames = []
if {"phenotypic_non_sparse", "phenotypic_sparse"} & enabled_benchmarks and not phenotypic_failures.empty:
    failure_frames.append(phenotypic_failures.assign(loader="phenotypic"))
if "demographics" in enabled_benchmarks and not demographics_failures.empty:
    failure_frames.append(demographics_failures.assign(loader="demographics"))
if "homotopic" in enabled_benchmarks and not homotopic_failures.empty:
    failure_frames.append(homotopic_failures.assign(loader="homotopic"))
if "weight_distance" in enabled_benchmarks and not weight_distance_failures.empty:
    failure_frames.append(weight_distance_failures.assign(loader="weight_distance"))
if "info_density" in enabled_benchmarks and not info_density_failures.empty:
    failure_frames.append(info_density_failures.assign(loader="info_density"))
if "reliability" in enabled_benchmarks and not reliability_failures.empty:
    failure_frames.append(reliability_failures.assign(loader="reliability"))

load_failures = (
    pd.concat(failure_frames, ignore_index=True)
    if failure_frames
    else pd.DataFrame(columns=["loader", "path", "error"])
)

print("Enabled benchmarks:", ", ".join(ENABLED_BENCHMARKS))
print("Metric block shapes:")
for name in ALL_KNOWN_BENCHMARKS:
    if name in enabled_benchmarks:
        print(f"  {name}: {benchmark_blocks[name].shape}")
    else:
        print(f"  {name}: excluded")
print("Combined score table:", combined_scores.shape)
print("Ranked score table:", ranked_scores.shape)
print("Ranking metrics:", len(metric_spec))

if not load_failures.empty:
    print("Loader warnings/failures:")
    display(load_failures.head(20))

display(metric_spec)
display(combined_scores.head(10))

Excluded degenerate combos from benchmark table: [['pyspi', 'bary_euclidean_mean'], ['pyspi', 'lmfit_Lasso']]
Enabled benchmarks: phenotypic_non_sparse, demographics, info_density, reliability
Metric block shapes:
  phenotypic_non_sparse: (181, 6)
  demographics: (181, 7)
  info_density: (180, 6)
  reliability: (182, 7)
Combined score table: (180, 22)
Ranked score table: (180, 59)
Ranking metrics: 10


,metric,rule_group,rule,weight,match,matched_pattern,minmax_col,maxnorm_col,qrank_col
0,demo_age_corr,demographics_age_corr,higher,0.25,exact,demo_age_corr,minmax_demo_age_corr,maxnorm_demo_age_corr,qrank_demo_age_corr
1,demo_age_mae,demographics_age_mae,lower,0.25,exact,demo_age_mae,minmax_demo_age_mae,maxnorm_demo_age_mae,qrank_demo_age_mae
2,demo_sex_acc,demographics_sex,higher,0.50,exact,demo_sex_acc,minmax_demo_sex_acc,maxnorm_demo_sex_acc,qrank_demo_sex_acc
3,discriminability,reliability_discriminability,higher,0.33,exact,discriminability,minmax_discriminability,maxnorm_discriminability,qrank_discriminability
4,gradient_similarity,reliability_gradient,higher,0.33,exact,gradient_similarity,minmax_gradient_similarity,maxnorm_gradient_similarity,qrank_gradient_similarity
5,mean_icc2,reliability_icc2,higher,0.33,exact,mean_icc2,minmax_mean_icc2,maxnorm_mean_icc2,qrank_mean_icc2
6,pheno_Cognition_corr,phenotypic_non_sparse_corr,higher,0.50,regex,^pheno_(?!sparse_).+_corr$,minmax_pheno_Cognition_corr,maxnorm_pheno_Cognition_corr,qrank_pheno_Cognition_corr
7,pheno_Cognition_r2,phenotypic_non_sparse_r2,higher,0.50,regex,^pheno_(?!sparse_).+_r2$,minmax_pheno_Cognition_r2,maxnorm_pheno_Cognition_r2,qrank_pheno_Cognition_r2
8,stable_rank,info_density_stable_rank,higher,0.50,exact,stable_rank,minmax_stable_rank,maxnorm_stable_rank,qrank_stable_rank
9,sv_entropy,info_density_sv_entropy,higher,0.50,exact,sv_entropy,minmax_sv_entropy,maxnorm_sv_entropy,qrank_sv_entropy


,method,func,lag,component,combo_key,is_directed,family,display_name,pyspi_category,pyspi_identifier,...,demo_sex_acc,discriminability,gradient_similarity,mean_icc2,pheno_Cognition_corr,pheno_Cognition_r2,stable_rank,sv_entropy,n_available_metrics,missing_metrics
0,pyspi,cov_EmpiricalCovariance,0,combined,pyspi__cov_EmpiricalCovariance,False,pyspi,PySPI cov_EmpiricalCovariance,basic,cov,...,0.9141,0.9893,0.6125,0.3571,0.4096,0.1574,2.0897,6.4114,10,
1,pyspi,cov_EllipticEnvelope,0,combined,pyspi__cov_EllipticEnvelope,False,pyspi,PySPI cov_EllipticEnvelope,basic,cov,...,0.9101,0.9873,0.5878,0.3329,0.4122,0.1594,2.0858,6.4173,10,
2,pyspi,cov_GraphicalLassoCV,0,combined,pyspi__cov_GraphicalLassoCV,False,pyspi,PySPI cov_GraphicalLassoCV,basic,cov,...,0.8901,0.9396,0.4956,0.4136,0.3617,0.1196,1.6897,6.2628,10,
3,pyspi,cov_LedoitWolf,0,combined,pyspi__cov_LedoitWolf,False,pyspi,PySPI cov_LedoitWolf,basic,cov,...,0.9141,0.9890,0.6125,0.3575,0.4096,0.1574,2.0881,6.4109,10,
4,pyspi,cov_MinCovDet,0,combined,pyspi__cov_MinCovDet,False,pyspi,PySPI cov_MinCovDet,basic,cov,...,0.9125,0.9880,0.5955,0.3335,0.4124,0.1593,2.0893,6.4156,10,
5,pyspi,cov_OAS,0,combined,pyspi__cov_OAS,False,pyspi,PySPI cov_OAS,basic,cov,...,0.9141,0.9890,0.6125,0.3575,0.4096,0.1574,2.0882,6.4111,10,
6,pyspi,cov_ShrunkCovariance,0,combined,pyspi__cov_ShrunkCovariance,False,pyspi,PySPI cov_ShrunkCovariance,basic,cov,...,0.9141,0.9893,0.6125,0.3571,0.4096,0.1574,2.0897,6.4114,10,
7,pyspi,prec_EmpiricalCovariance,0,combined,pyspi__prec_EmpiricalCovariance,False,pyspi,PySPI prec_EmpiricalCovariance,basic,prec,...,0.6518,0.5115,0.0124,0.0058,0.0591,-0.0086,4.2365,6.7391,10,
8,pyspi,prec_EllipticEnvelope,0,combined,pyspi__prec_EllipticEnvelope,False,pyspi,PySPI prec_EllipticEnvelope,basic,prec,...,0.6074,0.5077,0.0125,0.0036,0.0325,-0.0108,3.3286,6.6406,10,
9,pyspi,prec_GraphicalLassoCV,0,combined,pyspi__prec_GraphicalLassoCV,False,pyspi,PySPI prec_GraphicalLassoCV,basic,prec,...,0.9083,0.9792,0.0202,0.0984,0.3754,0.1297,19.6152,7.1484,10,


## Add Basic Validation Checks

Run simple assertions that catch duplicate rows, missing required fields, ranking polarity issues, and accidental prediction values on directional rows.

In [9]:
assert not combined_scores.empty, "Combined benchmark score table is empty."
assert combined_scores[KEY_COLS].notna().all().all(), "Some row keys are missing."
assert not combined_scores.duplicated(KEY_COLS).any(), "Duplicate method/func/lag/component rows found."
assert metric_cols, "No benchmark metric columns were detected."
assert ranked_scores["rank_n_available"].ge(0).all(), "Invalid rank coverage values."

enabled_benchmarks = set(ENABLED_BENCHMARKS)
phenotypic_metric_cols = [
    col
    for col in metric_cols
    if col.startswith("pheno_") or col.startswith("pheno_sparse_")
]
assert all("_Cognition_" in col for col in phenotypic_metric_cols), "Phenotypic metrics should be Cognition-only."
assert metric_spec["weight"].ge(0).all(), "Metric weights must be non-negative."

if "phenotypic_non_sparse" not in enabled_benchmarks:
    assert not any(col.startswith("pheno_") and not col.startswith("pheno_sparse_") for col in metric_cols), "Non-sparse phenotypic metrics should be excluded."
if "phenotypic_sparse" not in enabled_benchmarks:
    assert not any(col.startswith("pheno_sparse_") for col in metric_cols), "Sparse phenotypic metrics should be excluded."
if "demographics" not in enabled_benchmarks:
    assert not any(col.startswith("demo_") for col in metric_cols), "Demographic metrics should be excluded."
if "homotopic" not in enabled_benchmarks:
    assert "homotopic_rank_mean" not in metric_cols, "Homotopic metrics should be excluded."
if "weight_distance" not in enabled_benchmarks:
    assert "weight_distance_spearman" not in metric_cols, "Weight-distance metrics should be excluded."

INFO_DENSITY_METRICS = {
    "sv_entropy", "stable_rank", "trophic_incoherence_abs",
    "rich_club_auc", "rich_club_mean_rho", "rich_club_k_at_max",
    "rich_club_sig_k_range", "tsp_cost", "core_depth",
}
RELIABILITY_METRICS = {"mean_icc2", "mean_icc3", "gradient_similarity", "discriminability"}

if "info_density" not in enabled_benchmarks:
    assert not (INFO_DENSITY_METRICS & set(metric_cols)), "Info-density metrics should be excluded."
if "reliability" not in enabled_benchmarks:
    assert not (RELIABILITY_METRICS & set(metric_cols)), "Reliability metrics should be excluded."

# Intrinsic metrics should only appear on combined rows
intrinsic_metric_cols = [col for col in metric_cols if col in INFO_DENSITY_METRICS | RELIABILITY_METRICS]
if intrinsic_metric_cols:
    directional_intrinsic = combined_scores.loc[
        combined_scores["component"].isin(["upper", "lower"]),
        intrinsic_metric_cols,
    ]
    if not directional_intrinsic.empty:
        assert directional_intrinsic.isna().all().all(), "Intrinsic metrics should only appear on combined rows."

prediction_metric_cols = [
    col
    for col in metric_cols
    if col.startswith("pheno_") or col.startswith("pheno_sparse_") or col.startswith("demo_")
]
directional_prediction_values = combined_scores.loc[
    combined_scores["component"].isin(["upper", "lower"]),
    prediction_metric_cols,
]
if not directional_prediction_values.empty:
    assert directional_prediction_values.isna().all().all(), "Prediction metrics should only appear on combined rows."

for row in metric_spec.itertuples(index=False):
    finite_qrank = ranked_scores[row.qrank_col].dropna()
    if len(finite_qrank) >= 4:
        counts = finite_qrank.value_counts().reindex([0.0, 1.0, 2.0, 3.0], fill_value=0).astype(int)
        assert counts.max() - counts.min() <= 1, f"Uneven quantile bins for {row.metric}: {counts.to_dict()}"

    finite_maxnorm = ranked_scores[row.maxnorm_col].dropna()
    if len(finite_maxnorm) > 0 and row.weight > 0:
        assert np.isclose(finite_maxnorm.max(), 1.0), f"Maxnorm ranking should peak at 1.0 for {row.metric}."

if "demo_age_mae" in combined_scores.columns:
    finite_mae = combined_scores["demo_age_mae"].dropna()
    if len(finite_mae) > 1:
        oriented_mae = orient_values(finite_mae, "lower")
        assert oriented_mae.idxmax() == finite_mae.idxmin(), "Age MAE ranking orientation is wrong."

if "weight_distance_spearman" in combined_scores.columns:
    finite_wd = combined_scores["weight_distance_spearman"].dropna()
    if len(finite_wd) > 1:
        oriented_wd = orient_values(finite_wd, "absolute")
        assert np.isclose(oriented_wd.max(), finite_wd.abs().max()), "Weight-distance ranking should use absolute magnitude."

coverage = (
    combined_scores.groupby(["method", "component"], observed=True)
    .agg(
        n_rows=("func", "size"),
        mean_available_metrics=("n_available_metrics", "mean"),
        max_available_metrics=("n_available_metrics", "max"),
    )
    .reset_index()
)

display(coverage)
print("Validation checks passed.")

,method,component,n_rows,mean_available_metrics,max_available_metrics
0,ensemble,combined,9,10.0000,10
1,pyspi,combined,147,9.9456,10
2,skarf,combined,24,10.0000,10


Validation checks passed.


## Save Outputs

Write the raw, ranked, and long-format tables to disk, reload the saved files, and display compact leaderboard views.

In [10]:
combined_scores.to_csv(OUTPUT_RAW_PATH, index=False)
ranked_scores.to_csv(OUTPUT_RANKED_PATH, index=False)

id_cols = [
    "method",
    "func",
    "lag",
    "component",
    "combo_key",
    "is_directed",
    "family",
    "display_name",
    "pyspi_category",
    "pyspi_identifier",
]
long_scores = combined_scores.melt(
    id_vars=id_cols,
    value_vars=metric_cols,
    var_name="metric",
    value_name="score",
).dropna(subset=["score"])
long_scores.to_csv(OUTPUT_LONG_PATH, index=False)

raw_reload = pd.read_csv(OUTPUT_RAW_PATH)
ranked_reload = pd.read_csv(OUTPUT_RANKED_PATH)
assert raw_reload.shape == combined_scores.shape, "Raw saved CSV shape changed on reload."
assert ranked_reload.shape == ranked_scores.shape, "Ranked saved CSV shape changed on reload."

print("Saved raw table:", OUTPUT_RAW_PATH)
print("Saved ranked table:", OUTPUT_RANKED_PATH)
print("Saved long table:", OUTPUT_LONG_PATH)

leaderboard_cols = [
    "display_name",
    "method",
    "func",
    "lag",
    "component",
    "rank_n_available",
    "minmax_rank_sum",
    "minmax_rank_order",
    "maxnorm_rank_sum",
    "maxnorm_rank_order",
    "quantile_rank_sum",
    "quantile_rank_order",
]

combined_leaderboard = (
    ranked_scores.loc[ranked_scores["component"].eq("combined"), leaderboard_cols + metric_cols]
    .sort_values(["minmax_rank_sum", "maxnorm_rank_sum", "quantile_rank_sum"], ascending=False)
)

directional_view = (
    ranked_scores.loc[ranked_scores["component"].isin(["upper", "lower"]), leaderboard_cols + [
        col for col in ["homotopic_rank_mean", "weight_distance_spearman"] if col in ranked_scores.columns
    ]]
    .sort_values(["minmax_rank_sum", "maxnorm_rank_sum", "quantile_rank_sum"], ascending=False)
    .head(25)
)

quantile_counts = pd.DataFrame(
    [
        {
            "metric": row.metric,
            **ranked_scores[row.qrank_col]
            .dropna()
            .value_counts()
            .reindex([0.0, 1.0, 2.0, 3.0], fill_value=0)
            .astype(int)
            .rename({0.0: "q0", 1.0: "q1", 2.0: "q2", 3.0: "q3"})
            .to_dict(),
        }
        for row in metric_spec.itertuples(index=False)
    ]
)

print("Top combined rows:")
display(combined_leaderboard)
print("Top directional rows:")
display(directional_view)

print("Metric completeness:")
display(combined_scores[metric_cols].notna().sum().rename("n_rows_with_metric").to_frame())

print("Quantile bin counts:")
display(quantile_counts)

Saved raw table: /home/aerkent/skarf-experiments/results/hcp_1200_benchmark_scores_combined_analysis/combined_benchmark_scores.csv
Saved ranked table: /home/aerkent/skarf-experiments/results/hcp_1200_benchmark_scores_combined_analysis/combined_benchmark_scores_ranked.csv
Saved long table: /home/aerkent/skarf-experiments/results/hcp_1200_benchmark_scores_combined_analysis/combined_benchmark_scores_long.csv
Top combined rows:


,display_name,method,func,lag,component,rank_n_available,minmax_rank_sum,minmax_rank_order,maxnorm_rank_sum,maxnorm_rank_order,...,demo_age_corr,demo_age_mae,demo_sex_acc,discriminability,gradient_similarity,mean_icc2,pheno_Cognition_corr,pheno_Cognition_r2,stable_rank,sv_entropy
179,Ensemble top5_weighted_average,ensemble,top5_weighted_average,0,combined,10,3.4374,1,3.4362,1,...,0.3335,2.8579,0.9485,0.9939,0.9993,9.3880e-01,0.3874,0.1399,14.2684,7.0737
178,Ensemble top5_simple_average,ensemble,top5_simple_average,0,combined,10,3.4370,2,3.4357,2,...,0.3336,2.8576,0.9485,0.9939,0.9993,9.3872e-01,0.3871,0.1396,14.3085,7.0748
171,Ensemble top10_reference_decoder,ensemble,top10_reference_decoder,0,combined,10,3.4067,3,3.4177,3,...,0.3448,2.8247,0.9460,0.9971,0.9991,9.3021e-01,0.3969,0.1493,10.6343,6.9780
174,Ensemble top15_reference_decoder,ensemble,top15_reference_decoder,0,combined,10,3.3983,4,3.4114,4,...,0.3555,2.8160,0.9426,0.9977,0.9990,9.1807e-01,0.3952,0.1478,10.6317,6.9781
177,Ensemble top5_reference_decoder,ensemble,top5_reference_decoder,0,combined,10,3.3876,5,3.3956,5,...,0.3371,2.8437,0.9483,0.9952,0.9990,9.3532e-01,0.3912,0.1443,10.6466,6.9781
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,PySPI dswpli_multitaper_max_fs-1_fmin-0-25_fma...,pyspi,dswpli_multitaper_max_fs-1_fmin-0-25_fmax-0-5,0,combined,10,0.4357,176,0.8674,176,...,0.0415,10.3906,0.4527,0.5008,0.0155,2.1996e-03,-0.0544,-0.0080,1.0077,4.8207
118,PySPI dswpli_multitaper_max_fs-1_fmin-0_fmax-0-25,pyspi,dswpli_multitaper_max_fs-1_fmin-0_fmax-0-25,0,combined,10,0.3561,177,0.7958,177,...,0.0089,28.5329,0.4583,0.5002,0.0095,8.7077e-06,0.0491,-0.0069,1.0050,4.8183
117,PySPI dswpli_multitaper_max_fs-1_fmin-0_fmax-0-5,pyspi,dswpli_multitaper_max_fs-1_fmin-0_fmax-0-5,0,combined,10,0.3289,178,0.7646,178,...,-0.0138,28.6636,0.4583,0.5001,0.0076,1.7841e-05,0.0417,-0.0101,1.0050,4.8183
112,PySPI dspli_multitaper_max_fs-1_fmin-0_fmax-0-25,pyspi,dspli_multitaper_max_fs-1_fmin-0_fmax-0-25,0,combined,10,0.3025,179,0.7338,179,...,-0.0263,28.0614,0.4583,0.5011,0.0094,1.0205e-04,0.0152,-0.0100,1.0052,4.8048


Top directional rows:


,display_name,method,func,lag,component,rank_n_available,minmax_rank_sum,minmax_rank_order,maxnorm_rank_sum,maxnorm_rank_order,quantile_rank_sum,quantile_rank_order


Metric completeness:


,n_rows_with_metric
demo_age_corr,179
demo_age_mae,179
demo_sex_acc,179
discriminability,180
gradient_similarity,177
mean_icc2,180
pheno_Cognition_corr,179
pheno_Cognition_r2,179
stable_rank,180
sv_entropy,180


Quantile bin counts:


,metric,q0,q1,q2,q3
0,demo_age_corr,45,45,45,44
1,demo_age_mae,45,45,45,44
2,demo_sex_acc,45,45,45,44
3,discriminability,45,45,45,45
4,gradient_similarity,45,44,44,44
5,mean_icc2,45,45,45,45
6,pheno_Cognition_corr,45,45,45,44
7,pheno_Cognition_r2,45,45,45,44
8,stable_rank,45,45,45,45
9,sv_entropy,45,45,45,45


In [11]:
# Map each metric column to its benchmark group from METRIC_RULE_CONFIG
benchmark_metric_map: dict[str, list[str]] = {}
for col in metric_cols:
    rule_info = resolve_metric_rule_config(col)
    group = rule_info["rule_group"] if rule_info else "unmatched"
    benchmark_metric_map.setdefault(group, []).append(col)

print("Missing metrics per benchmark group (rows with NaN):\n")
for group, cols in sorted(benchmark_metric_map.items()):
    for col in cols:
        missing_mask = combined_scores[col].isna()
        n_missing = missing_mask.sum()
        n_total = len(combined_scores)
        missing_rows = combined_scores.loc[missing_mask, ["method", "func", "lag", "component"]]
        print(f"[{group}] {col}  —  {n_missing}/{n_total} missing")
        if n_missing > 0:
            display(missing_rows.reset_index(drop=True))
        print()


Missing metrics per benchmark group (rows with NaN):

[demographics_age_corr] demo_age_corr  —  1/180 missing


,method,func,lag,component
0,pyspi,tlmi_kraskov_NN-4,0,combined



[demographics_age_mae] demo_age_mae  —  1/180 missing


,method,func,lag,component
0,pyspi,tlmi_kraskov_NN-4,0,combined



[demographics_sex] demo_sex_acc  —  1/180 missing


...


[phenotypic_non_sparse_r2] pheno_Cognition_r2  —  1/180 missing


,method,func,lag,component
0,pyspi,tlmi_kraskov_NN-4,0,combined



[reliability_discriminability] discriminability  —  0/180 missing

[reliability_gradient] gradient_similarity  —  3/180 missing


,method,func,lag,component
0,pyspi,coint_aeg_tstat_trend-c_autolag-aic_maxlag-10,0,combined
1,pyspi,coint_aeg_tstat_trend-ct_autolag-aic_maxlag-10,0,combined
2,pyspi,coint_aeg_tstat_trend-ct_autolag-bic_maxlag-10,0,combined



[reliability_icc2] mean_icc2  —  0/180 missing



## Poster Leaderboard (Maxnorm Ranking)

Top 25 estimators by total maxnorm-weighted score, with per-group sub-ranks. Estimator labels match the x-axis conventions in `poster_figure2.ipynb`.

In [12]:
# Poster-style estimator label map (matches xlabels in poster_figure2.ipynb)
_POSTER_SKARF_DISPLAY = {
    "cov_empirical": "cov_emp",
    "cov_graphicallasso": "cov_glasso",
    "prec_empirical": "prec_emp",
    "prec_graphicallasso": "prec_glasso",
    "linear_ols": "lstsq_ols",
    "linear_ridge": "lstsq_ridge",
    "linear_lasso": "sparse_lasso",
    "linear_enet": "sparse_enet",
    "linear_lasso-pos": "sparse-pos_lasso",
    "linear_enet-pos": "sparse-pos_enet",
    "linear_pls": "lowrank_pls",
    "linear_pca-ridge": "lowrank_pca-ridge",
}


def _poster_label(method: str, func: str, lag: int) -> str:
    if method == "skarf":
        display = _POSTER_SKARF_DISPLAY.get(func, func)
        return f"skarf-{display}_lag_{int(lag)}"
    if method == "ensemble":
        return f"ensemble-{func}"
    return f"PySPI-{func}"


# Metric weights from metric_spec
_metric_weights = dict(zip(metric_spec["metric"], metric_spec["weight"]))

# Split metrics into extrinsic / intrinsic using rule_group prefix
_extrinsic_prefixes = ("phenotypic", "demographics")
_extrinsic_metrics = [
    row.metric
    for row in metric_spec.itertuples()
    if row.rule_group.startswith(_extrinsic_prefixes)
]
_intrinsic_metrics = [
    row.metric
    for row in metric_spec.itertuples()
    if not row.rule_group.startswith(_extrinsic_prefixes)
]


def _partial_maxnorm_sum(df: pd.DataFrame, subset: list[str]) -> pd.Series:
    if not subset:
        return pd.Series(0.0, index=df.index)
    return sum(df[f"maxnorm_{col}"].fillna(0.0) * _metric_weights[col] for col in subset)


# Work on combined rows only
_lb = ranked_scores[ranked_scores["component"] == "combined"].copy()
_lb["poster_label"] = [
    _poster_label(r.method, r.func, r.lag) for r in _lb.itertuples()
]
_lb["_extrinsic_sum"] = _partial_maxnorm_sum(_lb, _extrinsic_metrics)
_lb["_intrinsic_sum"] = _partial_maxnorm_sum(_lb, _intrinsic_metrics)
_lb["extrinsic_rank"] = _lb["_extrinsic_sum"].rank(method="min", ascending=False).astype(int)
_lb["intrinsic_rank"] = _lb["_intrinsic_sum"].rank(method="min", ascending=False).astype(int)

# Top 25 by total maxnorm score, sorted ascending (best at bottom)
_top25 = (
    _lb.nlargest(25, "maxnorm_rank_sum")
    .sort_values("maxnorm_rank_sum", ascending=False)
    .reset_index(drop=True)
)

# Column display map: internal name → (group header, metric header)
_col_map = [
    ("poster_label",         "FC Estimator"),
    ("pheno_Cognition_corr", "Behavioral Prediction -- Cognition -- Pearson's r -- (↑ better) -- (w=0.5)"),
    ("pheno_Cognition_r2",   "Behavioral Prediction -- Cognition -- R-squared -- (↑ better) -- (w=0.5)"),
    ("demo_sex_acc",         "Demographic Prediction -- Sex -- Accuracy -- (↑ better) -- (w=0.5)"),
    ("demo_age_corr",        "Demographic Prediction -- Age -- Pearson's r -- (↑ better) -- (w=0.25)"),
    ("demo_age_mae",         "Demographic Prediction -- Age -- MAE -- (↓ better) -- (w=0.25)"),
    ("extrinsic_rank",       "Extrinsic Rank"),
    ("sv_entropy",           "Structural Validity -- Complexity -- SVE (↑ better) -- (w=0.5)"),
    ("stable_rank",          "Structural Validity -- Complexity -- Stable Rank (↑ better) -- (w=0.5)"),
    ("mean_icc2",            "Stability Edgewise -- ICC2 (↑ better) -- (w=0.33)"),
    ("gradient_similarity",  "Stability Network -- Gradient (↑ better) -- (w=0.33)"),
    ("discriminability",     "Stability Discriminability -- SI (↑ better) -- (w=0.33)"),
    ("intrinsic_rank",       "Intrinsic Rank"),
    ("maxnorm_rank_order",   "Total Rank"),
]
_available = [(k, v) for k, v in _col_map if k in _top25.columns]

leaderboard_poster = _top25[[k for k, _ in _available]].copy()
leaderboard_poster.columns = [v for _, v in _available]

print(f"Poster leaderboard — top 25 of {len(_lb)} combined-component estimators")
print(f"Extrinsic metrics ({len(_extrinsic_metrics)}): {_extrinsic_metrics}")
print(f"Intrinsic metrics ({len(_intrinsic_metrics)}): {_intrinsic_metrics}")
display(leaderboard_poster)

Poster leaderboard — top 25 of 180 combined-component estimators
Extrinsic metrics (5): ['demo_age_corr', 'demo_age_mae', 'demo_sex_acc', 'pheno_Cognition_corr', 'pheno_Cognition_r2']
Intrinsic metrics (5): ['discriminability', 'gradient_similarity', 'mean_icc2', 'stable_rank', 'sv_entropy']


,FC Estimator,Behavioral Prediction -- Cognition -- Pearson's r -- (↑ better) -- (w=0.5),Behavioral Prediction -- Cognition -- R-squared -- (↑ better) -- (w=0.5),Demographic Prediction -- Sex -- Accuracy -- (↑ better) -- (w=0.5),Demographic Prediction -- Age -- Pearson's r -- (↑ better) -- (w=0.25),Demographic Prediction -- Age -- MAE -- (↓ better) -- (w=0.25),Extrinsic Rank,Structural Validity -- Complexity -- SVE (↑ better) -- (w=0.5),Structural Validity -- Complexity -- Stable Rank (↑ better) -- (w=0.5),Stability Edgewise -- ICC2 (↑ better) -- (w=0.33),Stability Network -- Gradient (↑ better) -- (w=0.33),Stability Discriminability -- SI (↑ better) -- (w=0.33),Intrinsic Rank,Total Rank
0,ensemble-top5_weighted_average,0.3874,0.1399,0.9485,0.3335,2.8579,45,7.0737,14.2684,0.9388,0.9993,0.9939,2,1
1,ensemble-top5_simple_average,0.3871,0.1396,0.9485,0.3336,2.8576,46,7.0748,14.3085,0.9387,0.9993,0.9939,1,2
2,ensemble-top10_reference_decoder,0.3969,0.1493,0.9460,0.3448,2.8247,28,6.9780,10.6343,0.9302,0.9991,0.9971,4,3
3,ensemble-top15_reference_decoder,0.3952,0.1478,0.9426,0.3555,2.8160,29,6.9781,10.6317,0.9181,0.9990,0.9977,5,4
4,ensemble-top5_reference_decoder,0.3912,0.1443,0.9483,0.3371,2.8437,39,6.9781,10.6466,0.9353,0.9990,0.9952,3,5
5,skarf-lowrank_pca-ridge_lag_0,0.4287,0.1746,0.9466,0.4288,2.8235,2,7.1016,14.9585,0.1509,0.7931,0.9998,16,6
6,ensemble-top10_weighted_average,0.3882,0.1420,0.9436,0.3549,2.8241,40,6.5857,2.3314,0.9287,0.9976,0.9969,8,7
7,ensemble-top10_simple_average,0.3881,0.1418,0.9439,0.3550,2.8239,42,6.5822,2.3231,0.9281,0.9976,0.9969,9,8
8,ensemble-top15_weighted_average,0.3863,0.1403,0.9405,0.3678,2.7955,43,6.5150,2.1774,0.9119,0.9966,0.9948,10,9
9,ensemble-top15_simple_average,0.3861,0.1401,0.9406,0.3677,2.7954,44,6.5130,2.1737,0.9111,0.9965,0.9947,11,10


## Poster Leaderboard Excluding Reliability Metrics (Ensemble Comparison)

To compare ensemble methods against PySPI/skarf on equal footing, this recomputes the same maxnorm-weighted poster-style ranking as above, restricted to the extrinsic (behavioral + demographic) and info-density metrics only.

In [13]:
_no_reliability_metrics = [
    row.metric for row in metric_spec.itertuples() if not row.rule_group.startswith("reliability")
]
_info_density_metrics_nr = [
    row.metric for row in metric_spec.itertuples() if row.rule_group.startswith("info_density")
]

_lb_nr = ranked_scores[ranked_scores["component"] == "combined"].copy()
_lb_nr["poster_label"] = [
    _poster_label(r.method, r.func, r.lag) for r in _lb_nr.itertuples()
]
_lb_nr["_extrinsic_sum"] = _partial_maxnorm_sum(_lb_nr, _extrinsic_metrics)
_lb_nr["_intrinsic_sum_nr"] = _partial_maxnorm_sum(_lb_nr, _info_density_metrics_nr)
_lb_nr["extrinsic_rank"] = _lb_nr["_extrinsic_sum"].rank(method="min", ascending=False).astype(int)
_lb_nr["intrinsic_rank_no_reliability"] = _lb_nr["_intrinsic_sum_nr"].rank(method="min", ascending=False).astype(int)
_lb_nr["no_reliability_rank_sum"] = _partial_maxnorm_sum(_lb_nr, _no_reliability_metrics)
_lb_nr["no_reliability_rank_order"] = _lb_nr["no_reliability_rank_sum"].rank(method="min", ascending=False).astype(int)

_top25_nr = (
    _lb_nr.nlargest(25, "no_reliability_rank_sum")
    .sort_values("no_reliability_rank_sum", ascending=False)
    .reset_index(drop=True)
)

_col_map_nr = [
    ("poster_label",         "FC Estimator"),
    ("pheno_Cognition_corr", "Behavioral Prediction -- Cognition -- Pearson's r -- (↑ better) -- (w=0.5)"),
    ("pheno_Cognition_r2",   "Behavioral Prediction -- Cognition -- R-squared -- (↑ better) -- (w=0.5)"),
    ("demo_sex_acc",         "Demographic Prediction -- Sex -- Accuracy -- (↑ better) -- (w=0.5)"),
    ("demo_age_corr",        "Demographic Prediction -- Age -- Pearson's r -- (↑ better) -- (w=0.25)"),
    ("demo_age_mae",         "Demographic Prediction -- Age -- MAE -- (↓ better) -- (w=0.25)"),
    ("extrinsic_rank",       "Extrinsic Rank"),
    ("sv_entropy",           "Structural Validity -- Complexity -- SVE (↑ better) -- (w=0.5)"),
    ("stable_rank",          "Structural Validity -- Complexity -- Stable Rank (↑ better) -- (w=0.5)"),
    ("intrinsic_rank_no_reliability", "Intrinsic Rank (no reliability)"),
    ("no_reliability_rank_order",     "Total Rank (no reliability)"),
]
_available_nr = [(k, v) for k, v in _col_map_nr if k in _top25_nr.columns]

leaderboard_poster_no_reliability = _top25_nr[[k for k, _ in _available_nr]].copy()
leaderboard_poster_no_reliability.columns = [v for _, v in _available_nr]

n_ensemble_in_top25 = int((_top25_nr["method"] == "ensemble").sum())
print(f"Poster leaderboard (no reliability) — top 25 of {len(_lb_nr)} combined-component estimators")
print(f"Excluded metrics (reliability): {[m for m in metric_cols if m not in _no_reliability_metrics]}")
print(f"Included metrics ({len(_no_reliability_metrics)}): {_no_reliability_metrics}")
print(f"Ensemble combos in top 25: {n_ensemble_in_top25}")
display(leaderboard_poster_no_reliability)

Poster leaderboard (no reliability) — top 25 of 180 combined-component estimators
Excluded metrics (reliability): ['discriminability', 'gradient_similarity', 'mean_icc2']
Included metrics (7): ['demo_age_corr', 'demo_age_mae', 'demo_sex_acc', 'pheno_Cognition_corr', 'pheno_Cognition_r2', 'stable_rank', 'sv_entropy']
Ensemble combos in top 25: 5


,FC Estimator,Behavioral Prediction -- Cognition -- Pearson's r -- (↑ better) -- (w=0.5),Behavioral Prediction -- Cognition -- R-squared -- (↑ better) -- (w=0.5),Demographic Prediction -- Sex -- Accuracy -- (↑ better) -- (w=0.5),Demographic Prediction -- Age -- Pearson's r -- (↑ better) -- (w=0.25),Demographic Prediction -- Age -- MAE -- (↓ better) -- (w=0.25),Extrinsic Rank,Structural Validity -- Complexity -- SVE (↑ better) -- (w=0.5),Structural Validity -- Complexity -- Stable Rank (↑ better) -- (w=0.5),Intrinsic Rank (no reliability),Total Rank (no reliability)
0,skarf-lstsq_ridge_lag_0,0.4233,0.1573,0.9350,0.3649,2.9255,11,7.2072,29.6209,5,1
1,PySPI-prec_ShrunkCovariance,0.4130,0.1560,0.9450,0.3263,2.9783,21,7.2888,26.5836,6,2
2,PySPI-prec_LedoitWolf,0.3799,0.1296,0.9290,0.2468,3.0734,61,7.3239,34.5367,2,3
3,skarf-lowrank_pca-ridge_lag_0,0.4287,0.1746,0.9466,0.4288,2.8235,2,7.1016,14.9585,15,4
4,PySPI-prec_OAS,0.3768,0.1275,0.9291,0.2434,3.0767,63,7.3245,34.8343,1,5
5,skarf-sparse_enet_lag_0,0.3486,0.1088,0.9106,0.3914,2.8711,67,7.2352,33.5269,4,6
6,skarf-lstsq_ridge_lag_1,0.4385,0.1788,0.9446,0.4315,2.8042,1,7.0609,10.3732,18,7
7,skarf-sparse_lasso_lag_0,0.3352,0.0982,0.9028,0.3814,2.8855,71,7.2365,34.7756,3,8
8,skarf-sparse_enet_lag_1,0.3881,0.1393,0.9166,0.4047,2.8576,38,7.1868,20.5180,8,9
9,skarf-lowrank_pls_lag_0,0.3961,0.1449,0.9326,0.3865,2.8880,27,7.1438,18.2854,13,10
